<a href="https://colab.research.google.com/github/ronniedebojit2002-netizen/Data-Intelligence-System/blob/main/Part4_LLM_Feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install jsonschema -q

In [21]:
import os
import json
import re
import requests
import joblib
import pandas as pd
import numpy as np

from jsonschema import validate, ValidationError

print("Libraries imported successfully.")

Libraries imported successfully.


In [22]:
from getpass import getpass

os.environ["LLM_API_KEY"] = getpass("Enter your OpenRouter API Key: ")

print("API key stored successfully.")

Enter your OpenRouter API Key: ··········
API key stored successfully.


In [23]:
print("API key loaded:", "LLM_API_KEY" in os.environ)

API key loaded: True


In [24]:
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    api_key = os.environ["LLM_API_KEY"]

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "openai/gpt-4.1-mini",
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None

    return response.json()["choices"][0]["message"]["content"]

In [25]:
system_prompt = "You are a helpful assistant."

user_prompt = "Reply with only the word: hello"

reply = call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

print(reply)

hello


In [26]:
def has_pii(text):
    email_pattern = r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"

    phone_pattern = r"\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b"

    return bool(
        re.search(email_pattern, text)
        or
        re.search(phone_pattern, text)
    )

In [27]:
print(has_pii("My email is student@gmail.com"))

print(has_pii("House price prediction"))

True
False


In [28]:
model = joblib.load("best_model.pkl")

print("Model loaded successfully.")

Model loaded successfully.


In [29]:
housing_df = pd.read_csv("cleaned_data.csv")

housing_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [30]:
# Remove target
X = housing_df.drop(columns=["SalePrice"])

# Ordinal mapping
quality_map = {
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

ordinal_features = [
    "ExterQual",
    "ExterCond",
    "KitchenQual",
    "HeatingQC"
]

X_encoded = X.copy()

for col in ordinal_features:
    if col in X_encoded.columns:
        X_encoded[col] = X_encoded[col].map(quality_map)

remaining = X_encoded.select_dtypes(
    include=["object", "category"]
).columns

X_encoded = pd.get_dummies(
    X_encoded,
    columns=remaining,
    drop_first=True
)

print(X_encoded.shape)

(1460, 235)


In [31]:
sample1 = X_encoded.iloc[[0]]
sample2 = X_encoded.iloc[[100]]
sample3 = X_encoded.iloc[[500]]

samples = [sample1, sample2, sample3]

In [32]:
schema = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string"},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

In [33]:
system_prompt = """
You are an AI model explanation assistant.

Given:
- feature values
- predicted class
- prediction probability

Return ONLY valid JSON.

The JSON must contain exactly these fields:

{
"prediction_label":"",
"confidence_level":"",
"top_reason":"",
"second_reason":"",
"next_step":""
}

Do not include markdown.
Do not include explanations outside JSON.
"""

In [34]:
def explain_prediction(features):

    prediction = model.predict(features)[0]

    probability = model.predict_proba(features)[0].max()

    feature_json = features.iloc[0].to_dict()

    user_prompt = f"""
Feature values:

{json.dumps(feature_json, indent=2)}

Predicted Class:

{prediction}

Prediction Probability:

{probability:.4f}
"""

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None

    response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    print("Raw Response:\n")
    print(response)

    try:

        result = json.loads(response.strip())

        validate(result, schema)

        print("\nValidation Passed ✅")

        return result

    except json.JSONDecodeError as e:

        print("JSON Error:", e)

        return {
            "prediction_label": None,
            "confidence_level": None,
            "top_reason": None,
            "second_reason": None,
            "next_step": None
        }

    except ValidationError as e:

        print("Validation Error:", e)

        return {
            "prediction_label": None,
            "confidence_level": None,
            "top_reason": None,
            "second_reason": None,
            "next_step": None
        }

In [35]:
for i, sample in enumerate(samples, start=1):

    print("=" * 60)
    print(f"Sample {i}")
    print("=" * 60)

    explanation = explain_prediction(sample)

    print("\nValidated JSON")

    print(explanation)

    print()

Sample 1
Raw Response:

{
  "prediction_label": "1",
  "confidence_level": "97.83%",
  "top_reason": "High Overall Quality (OverallQual=7) indicating a well-built house",
  "second_reason": "Good Basement Quality and Finish (BsmtQual_Gd and BsmtFinType1_GLQ) contributing to higher value",
  "next_step": "Consider verifying the condition of the house and neighborhood factors to confirm the prediction"
}

Validation Passed ✅

Validated JSON
{'prediction_label': '1', 'confidence_level': '97.83%', 'top_reason': 'High Overall Quality (OverallQual=7) indicating a well-built house', 'second_reason': 'Good Basement Quality and Finish (BsmtQual_Gd and BsmtFinType1_GLQ) contributing to higher value', 'next_step': 'Consider verifying the condition of the house and neighborhood factors to confirm the prediction'}

Sample 2
Raw Response:

{
  "prediction_label": "1",
  "confidence_level": "High",
  "top_reason": "OverallQual of 6 indicates above average quality",
  "second_reason": "GarageCars of 2

In [36]:
def explain_prediction_temp(features, temperature):

    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0].max()

    feature_json = features.iloc[0].to_dict()

    user_prompt = f"""
Feature values:
{json.dumps(feature_json, indent=2)}

Predicted Class:
{prediction}

Prediction Probability:
{probability:.4f}
"""

    response = call_llm(
        system_prompt,
        user_prompt,
        temperature=temperature
    )

    print(f"\nTemperature = {temperature}")
    print(response)

In [37]:
for sample in samples:

    explain_prediction_temp(sample, 0)

    explain_prediction_temp(sample, 0.7)

    print("="*80)


Temperature = 0
{
  "prediction_label": "1",
  "confidence_level": "97.83%",
  "top_reason": "High Overall Quality (OverallQual=7) indicating a well-built house",
  "second_reason": "Good Basement Quality and Finish (BsmtQual=Gd, BsmtFinType1=GLQ) adding value",
  "next_step": "Proceed with detailed inspection and valuation to confirm predicted class"
}

Temperature = 0.7
{
  "prediction_label": "1",
  "confidence_level": "97.83%",
  "top_reason": "High Overall Quality (OverallQual = 7) indicating good condition and build quality",
  "second_reason": "Large Garage Area (GarageArea = 548) and 2 Garage Cars indicating ample parking and storage",
  "next_step": "Consider further inspection or detailed appraisal to confirm the valuation and condition"
}

Temperature = 0
{
  "prediction_label": "1",
  "confidence_level": "High",
  "top_reason": "OverallQual of 6 indicates above average quality contributing to higher predicted class",
  "second_reason": "GarageCars of 2 and GarageArea of 48

In [38]:
blocked = "Customer email is student@gmail.com"

if has_pii(blocked):
    print("Input blocked: PII detected.")
else:
    print(call_llm(system_prompt, blocked))

Input blocked: PII detected.


In [39]:
clean = "This house has a garage and four bedrooms."

if has_pii(clean):
    print("Input blocked: PII detected.")
else:
    print(call_llm(system_prompt, clean))

{
  "prediction_label": "Family Home",
  "confidence_level": "High",
  "top_reason": "Presence of four bedrooms indicates suitability for a family.",
  "second_reason": "Garage adds convenience and is common in family homes.",
  "next_step": "Recommend scheduling a viewing to assess suitability for family needs."
}
